In [ ]:
# 필요한 도구 설치
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 한글 깨짐 방지
rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리

In [ ]:
# 데이터 불러오기
file_name = "파일명.xlsx"
df = pd.read_excel(file_name)

# 파이썬이 읽을 수 있도록 날짜 형식 변환
df['date'] = pd.to_datetime(df['date'])

# 주기 데이터일 경우, 월 별 데이터로 전환하는 코드 ( 현버전, 구버전 )
try:
    monthly_data = df.set_index('date').resample('ME')['value'].sum().reset_index()
except ValueError:
    monthly_data = df.set_index('date').resample('M')['value'].sum().reset_index()

# 컬럼 정리
monthly_final.columns = ['month', 'value']

# ADF 검정

In [ ]:
# 데이터 지정
target_data = month_data['value']

# 그래프 그리기
plt.figure(figsize=(12, 5))
plt.plot(month_data['month'], target_data, color='green', label='검정할 데이터')
plt.title("ADF 검정", fontsize=15)
plt.xlabel('기간')
plt.ylabel('값')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

# ADF 검정
result = adfuller(target_data.dropna())
print(f"ADF 검정 결과")
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.5f}")

# 로그 변환

In [ ]:
# 로그 변환 ( 분산 안정화 ) 
# 새로운 컬럼 생성 뒤 로그 변환
monthly_data['log_value'] = np.log1p(monthly_data['value'])

# 그래프 그리기
plt.figure(figsize=(12, 6))
plt.plot(monthly_data['month'], monthly_data['log_value'], marker='o')
plt.title("로그 변환 후 데이터")
plt.xlabel("기간")
plt.ylabel("로그 값")
plt.grid(True)
plt.xticks(rotation=45)
plt.show()

# 모델 학습

In [ ]:
# SARIMA 모델 학습
model = SARIMAX(monthly_data['log_value'],
                order=(p, d, q),             
                seasonal_order=(P, D, Q, s),  
                enforce_stationarity=False,
                enforce_invertibility=False)

results = model.fit(disp=False)
print(results.summary())

# 잔차 시계열 그래프

In [ ]:
# 잔차 시계열 그래프 출력
# 잔차의 퍼짐 정도를 한 눈에 알아보기 위해 시각화
residuals = results.resid

plt.figure(figsize=(12, 6))
plt.plot(monthly_final['month'], residuals, label='잔차 (Residuals)')
plt.axhline(y=0, color='r', linestyle='--', linewidth=1.5)
plt.title('잔차 시계열', fontsize=15)
plt.xlabel('기간')
plt.ylabel('잔차 값')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

# 잔차의 ACF / PACF

In [ ]:
# 잔차의 ACF 시도표
plt.figure(figsize=(12, 8))
plt.subplot(2, 1, 1)
plot_acf(residuals, lags=24, ax=plt.gca(), title="잔차의 ACF 시도표")
plt.grid(True, alpha=0.3)

# 잔차의 PACF 시도표
plt.subplot(2, 1, 2)
plot_pacf(residuals, lags=24, ax=plt.gca(), title="잔차의 PACF 시도표")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 잔차의 모수 유의성 검정

In [ ]:
# 잔차의 모수 유의성 검정 + 포트맨토 검정
print("모수 유의성 검정")
print(results.summary().tables[1])

print("잔차 포트맨토 검정 (Ljung-Box)")
lb_result = acorr_ljungbox(results.resid, lags=[12], return_df=True)

print(lb_result)

# 미래 예측

In [ ]:
# 11 예측
forecast = results.get_forecast(steps=30)          # 예측 기간 수정
pred_log = forecast.predicted_mean                 # 예측의 평균치 뽑기
conf_int_log = forecast.conf_int()                 # 예측의 신뢰구간 범위 가져오기

# 최솟값, 최댓값을 포함하여 로그를 씌웠던 데이터를 실제 값으로 되돌림
pred_value   = np.expm1(pred_log)
lower_limits = np.expm1(conf_int_log.iloc[:, 0])
upper_limits = np.expm1(conf_int_log.iloc[:, 1])

# 음수의 값이 존재하지 않을 경우, 0으로 조정 필요
pred_value[pred_value < 0] = 0
lower_limits[lower_limits < 0] = 0
upper_limits[upper_limits < 0] = 0

# 미래 날짜 데이터 생성
last_date = monthly_data['month'].iloc[-1]
prediction_dates = pd.date_range(start=last_date, periods=31, freq='ME')[1:]

# 미래 예측 데이터 그리기
plt.figure(figsize=(14, 7))
plt.plot(monthly_data['month'], np.expm1(monthly_data['log_value']),                # 과거 실제 데이터 그리기
         label='실제 값 (History)', color='blue', linewidth=2)

plt.plot(prediction_dates, pred_value,
         label='예측 값 (Forecast)', color='red', linestyle='--', marker='o')       # 미래 예측 데이터 이어서 그리기

plt.fill_between(prediction_dates, lower_limits, upper_limits,                     # 신뢰 구간 표시
                 color='gray', alpha=0.2, label='95% 신뢰구간')

# 데이터 그리기
plt.title("예측 결과", fontsize=16)
plt.xlabel("기간")
plt.ylabel("값")
plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# RMSE / MAE 확인

In [ ]:
# 음수값이 없을 경우, 0보다 큰 데이터만 골라내기
mask   = monthly_data['value'] > 0
actual = monthly_data.loc[mask, 'value']

# 로그 값을 실제 값으로 변환
fitted = np.expm1(results.fittedvalues[mask])

# 차분할 경우, 공통된 부분 찾기
common_idx = actual.index.intersection(fitted.index)
actual = actual.loc[common_idx]
fitted = fitted.loc[common_idx]

# RMSE / MAE 출력
rmse = np.sqrt(mean_squared_error(actual, fitted))
mae  = mean_absolute_error(actual, fitted)
print(f"RMSE : {rmse:,.0f}")
print(f"MAE  : {mae:,.0f}")